# Chatterbox frente a Qwen3-TTS

Genera las **mismas frases** con los dos motores, con **tu misma voz de referencia**, y las mide igual:
WER con Whisper, ritmo frente al modelo de duración del narrador y vocalizaciones sobrantes (los "oh"
pegados a la voz). Después las pones a la par y decides con el oído.

**Antes de empezar**, crea el entorno de Qwen (una vez):

```bash
./env/setup_qwen.sh
```

Va en un venv aparte (`.venv-qwen`) porque Chatterbox fija `transformers==5.2.0` y `qwen-tts` trae la
suya. Así el pipeline sigue funcionando mientras comparas.

Este notebook se ejecuta con el kernel del proyecto (`series-redes-neuronales (.venv 3.13)`):
genera Chatterbox aquí y llama al otro venv por subproceso, igual que hace el pipeline con sus etapas.

Qwen3-TTS necesita, además del audio de referencia, **su transcripción**. No hace falta que la
escribas: se obtiene con Whisper a partir de `narrador.wav`.

In [ ]:
import gc, json, os, subprocess, sys
from pathlib import Path

RAIZ = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / "pipeline" / "produce.py").exists()), None)
if RAIZ is None:
    raise RuntimeError("Abre el notebook desde dentro de series-redes-neuronales/")
if Path(sys.prefix).resolve() != (RAIZ / ".venv").resolve():
    raise RuntimeError(f"Kernel equivocado: {sys.executable}. "
                       f"Selecciona 'series-redes-neuronales (.venv 3.13)' o {RAIZ / '.venv/bin/python'}")
sys.path.insert(0, str(RAIZ))

import jiwer
import pandas as pd
import soundfile as sf
import torch
from IPython.display import Audio, HTML, display

from lib import audio, config
from lib.texto import Frase, normalizar_para_wer, numeros_a_palabras, terminos_pronunciacion, texto_para_tts
from pipeline.e2_tts import Oido, Voz, detectar_sobrantes, limpiar_bordes

PYTHON_QWEN = RAIZ / ".venv-qwen" / "bin" / "python"
if not PYTHON_QWEN.exists():
    raise RuntimeError("Falta .venv-qwen: ejecuta ./env/setup_qwen.sh")

SALIDA = RAIZ / "build" / "comparacion_tts"
voz_cfg = config.cargar("voz")
qa = config.cargar("pipeline")["voz_qa"]
REFERENCIA = RAIZ / voz_cfg["referencia"]
print(RAIZ, "·", torch.__version__, "·", torch.cuda.get_device_name(0))

## Frases de prueba

Las 10 de la calibración más los casos que ya nos han dado problemas con Chatterbox: el acento inglés en «red» y las frases donde aparecían vocalizaciones sobrantes.

In [ ]:
nb = json.loads((RAIZ / "notebooks/01_calibrar_voz.ipynb").read_text(encoding="utf-8"))
src = "".join("".join(c["source"]) for c in nb["cells"])
ns = {}
exec(src[src.index("FRASES = ["):src.index("# A = valores")], ns)

FRASES = [{"id": f"f{i:02d}", "texto": t} for i, t in enumerate(ns["FRASES"], 1)]
FRASES += [
    {"id": "r01", "texto": "Y sin embargo, una red neuronal encontrará la frontera ella sola."},
    {"id": "r02", "texto": "La red recibe el resultado, lo redondea rápidamente y reduce el ruido."},
    {"id": "r03", "texto": "Hijueputa que rico."}, # Ninguna línea recta puede separarlos
    {"id": "r04", "texto": "Los puntos no se han movido por su cuenta: es el tapiz de debajo el que se reordena."},
]
REGENERAR = True  # ponlo en False solo si quieres reutilizar los wav antiguos
print(f"{len(FRASES)} frases · referencia: {REFERENCIA.relative_to(RAIZ)}")

## 1. Transcribir la referencia (la necesita Qwen) y generar con Chatterbox

In [ ]:
oido = Oido(qa["modelo_whisper"])
texto_referencia, _ = oido.transcribir(REFERENCIA)
print("transcripción de la referencia:", texto_referencia)

voz = Voz(voz_cfg)
dir_chatterbox = SALIDA / "chatterbox"
dir_chatterbox.mkdir(parents=True, exist_ok=True)
for frase in FRASES:
    ruta = dir_chatterbox / f"{frase['id']}.wav"
    if ruta.exists() and not REGENERAR:
        continue
    if ruta.exists():
        ruta.unlink()
    torch.manual_seed(voz_cfg["semilla"])
    wav, sr = voz.sintetizar(texto_para_tts(frase["texto"]), voz_cfg["semilla"])
    wav = audio.normalizar_sonoridad(audio.recortar_silencios(wav, sr), sr)
    sf.write(ruta, wav, sr)

print("Chatterbox listo")
del voz; gc.collect(); torch.cuda.empty_cache()   # libera la GPU para Qwen

## 2. Generar con Qwen3-TTS (en su propio venv)

La primera vez descarga el modelo (~3 GB). Si prefieres el modelo pequeño, cambia `MODELO` por `Qwen/Qwen3-TTS-12Hz-0.6B-Base`.

In [ ]:
MODELO = "Qwen/Qwen3-TTS-12Hz-1.7B-Base"

dir_qwen = SALIDA / "qwen"
trabajo = SALIDA / "trabajo_qwen.json"
SALIDA.mkdir(parents=True, exist_ok=True)
if REGENERAR:
    for frase in FRASES:
        ruta = dir_qwen / f"{frase['id']}.wav"
        if ruta.exists():
            ruta.unlink()
frases_qwen = [f for f in FRASES if REGENERAR or not (dir_qwen / f"{f['id']}.wav").exists()]
trabajo.write_text(json.dumps({
    "referencia": str(REFERENCIA), "texto_referencia": texto_referencia, "modelo": MODELO,
    "idioma": "Spanish", "salida": str(dir_qwen),
    "frases": frases_qwen,
}, ensure_ascii=False, indent=1), encoding="utf-8")

# Sin HF_HUB_OFFLINE: el pipeline lo activa al importar e2_tts y Qwen aún tiene que descargar.
entorno = {k: v for k, v in os.environ.items() if k != "HF_HUB_OFFLINE"}
proceso = subprocess.run([str(PYTHON_QWEN), str(RAIZ / "notebooks/generar_qwen.py"), str(trabajo)],
                         cwd=RAIZ, env=entorno, capture_output=True, text=True)
print(proceso.stdout[-2000:])
if proceso.returncode != 0:
    print(proceso.stderr[-3000:])
    raise RuntimeError("Qwen3-TTS falló; revisa el error de arriba")
if proceso.returncode != 0:

## 3. Medir los dos con el mismo rasero

El mismo QA del pipeline: WER excluyendo el diccionario de pronunciación, ritmo frente al modelo de duración del narrador y detección de vocalizaciones pegadas a la voz.

In [ ]:
excluir = terminos_pronunciacion()
a_modelo, b_modelo = qa["duracion_modelo"]

filas = []
for motor, carpeta in (("Chatterbox", SALIDA / "chatterbox"), ("Qwen3-TTS", SALIDA / "qwen")):
    for frase in FRASES:
        ruta = carpeta / f"{frase['id']}.wav"
        if not ruta.exists():
            continue
        transcripcion, palabras = oido.transcribir(ruta)
        wav, sr = sf.read(ruta, dtype="float32")
        wav, _, cola = limpiar_bordes(wav, sr, palabras)     # sonidos sueltos tras un silencio
        sobrantes = detectar_sobrantes(Frase(frase["texto"]), palabras)   # "oh" pegado a la voz
        duracion = len(wav) / sr
        caracteres = len(numeros_a_palabras(frase["texto"]))
        filas.append(dict(
            motor=motor, id=frase["id"], texto=frase["texto"], ruta=ruta, duracion=round(duracion, 2),
            wer=round(jiwer.wer(normalizar_para_wer(frase["texto"], excluir),
                                normalizar_para_wer(transcripcion, excluir)), 3),
            ritmo=round(duracion / (a_modelo + b_modelo * caracteres), 2),
            sobrantes=sobrantes, cola=cola, transcripcion=transcripcion))

res = pd.DataFrame(filas)
del oido; gc.collect(); torch.cuda.empty_cache()

resumen = res.groupby("motor").agg(
    wer_medio=("wer", "mean"), wer_peor=("wer", "max"),
    frases_con_wer_alto=("wer", lambda w: int((w > qa["wer_max_frase"]).sum())),
    ritmo_medio=("ritmo", "mean"),
    con_vocalizacion=("sobrantes", lambda s: int((s != "").sum())),
    con_cola=("cola", lambda s: int((s != "").sum())),
).round(3)
display(resumen)

In [ ]:
# Dónde falla cada uno: frases con WER alto, vocalizaciones o colas.
problemas = res[(res.wer > qa["wer_max_frase"]) | (res.sobrantes != "") | (res.cola != "")]
display(problemas[["motor", "id", "wer", "ritmo", "sobrantes", "cola", "transcripcion"]]
        if len(problemas) else "Ninguno de los dos motores da problemas medibles 🎉")

## 4. Escuchar en paralelo

Lo que las cifras no miden: el acento (fíjate en «red», «resultado», «rápidamente»), la naturalidad y si se parece a ti. Escucha la misma frase en los dos motores, seguidas.

In [ ]:
for frase in FRASES:
    display(HTML(f"<h4>{frase['id']} · {frase['texto']}</h4>"))
    for motor in ("Chatterbox", "Qwen3-TTS"):
        fila = res[(res.motor == motor) & (res.id == frase["id"])]
        if fila.empty:
            continue
        fila = fila.iloc[0]
        print(f"{motor:11s} WER {fila.wer:.2f} · ritmo {fila.ritmo:.2f}"
              + (f" · sobrantes: {fila.sobrantes}" if fila.sobrantes else "")
              + (f" · cola: {fila.cola}" if fila.cola else ""))
        display(Audio(str(fila.ruta)))

## 5. Decidir

**Si gana Chatterbox:** no hay nada que hacer; el pipeline sigue igual.

**Si gana Qwen3-TTS**, el cambio afecta a una sola clase del pipeline (`Voz`, en `pipeline/e2_tts.py`),
y hay que:

1. Sustituir esa clase por una que llame a Qwen (el resto (QA con Whisper, marcas, recortes, montaje) no cambia).
2. Decidir si el pipeline usa el venv de Qwen para la etapa 2 o si se migra entero.
3. Subir `PROCESADO` en `e2_tts.py` para que se regenere todo el audio.
4. Recalibrar: el ritmo del narrador cambia, así que hay que rehacer `voz_qa.duracion_modelo`
   con `.venv/bin/python -m lib.ritmo`.
5. Revisar si siguen haciendo falta `config/pronunciacion.yaml` y la regla `erre_inicial`.

Anota aquí abajo lo que observes, para tenerlo por escrito cuando decidas.

In [ ]:
# Notas de la escucha (rellénalas tú):
NOTAS = """
Chatterbox: ...
Qwen3-TTS:  ...
Decisión:   ...
"""
(SALIDA / "notas.md").write_text(NOTAS, encoding="utf-8")
print(NOTAS)